# Derive Andvari Network Enforcement Summary

This notebook derives a compact proxy/firewall analysis artifact from `andvari_invocations.csv`.
The derived CSV is intended for the small thesis results subsection on proxy and firewall enforcement events.

It produces:

- `andvari_network_enforcement_summary.csv`, with one row per Andvari invocation.
- A compact table grouped by agent and reconstruction variant.
- A LaTeX table matching the thesis result presentation.


In [3]:
from pathlib import Path
import pandas as pd

INPUT_CSV = Path('exports/heimdall-analysis-20260520T203812Z/tables/andvari_invocations.csv')
OUTPUT_CSV = Path('andvari_network_enforcement_summary.csv')

df = pd.read_csv(INPUT_CSV)
df.head()


,agent,blocked_egress_line_count,proxy_access_line_count,proxy_denied_count,reason,repo_slug,run_id,status,step,variant
0,codex,0,216,0,NaN,ulisesbocchio_jasypt-spring-boot,20260505T122153Z__ulisesbocchio_jasypt-spring-...,passed,andvari,generated
1,codex,0,222,0,NaN,ulisesbocchio_jasypt-spring-boot,20260505T122153Z__ulisesbocchio_jasypt-spring-...,passed,andvari-v2,v2
2,codex,0,210,0,NaN,ulisesbocchio_jasypt-spring-boot,20260505T122153Z__ulisesbocchio_jasypt-spring-...,passed,andvari-v3,v3
3,codex,0,156,0,NaN,mbechler_marshalsec,20260505T150455Z__mbechler_marshalsec__243c5aa8,passed,andvari,generated
4,codex,0,156,0,NaN,mbechler_marshalsec,20260505T150455Z__mbechler_marshalsec__243c5aa8,passed,andvari-v2,v2


## Validate required columns

The analysis only uses columns that already exist in `andvari_invocations.csv`. The derived fields are computed from the proxy and firewall counts.

In [4]:
required = [
    'agent',
    'blocked_egress_line_count',
    'proxy_access_line_count',
    'proxy_denied_count',
    'reason',
    'repo_slug',
    'run_id',
    'status',
    'step',
    'variant',
]

missing = [col for col in required if col not in df.columns]
if missing:
    raise ValueError(f'Missing required columns: {missing}')

df[required].head()


,agent,blocked_egress_line_count,proxy_access_line_count,proxy_denied_count,reason,repo_slug,run_id,status,step,variant
0,codex,0,216,0,NaN,ulisesbocchio_jasypt-spring-boot,20260505T122153Z__ulisesbocchio_jasypt-spring-...,passed,andvari,generated
1,codex,0,222,0,NaN,ulisesbocchio_jasypt-spring-boot,20260505T122153Z__ulisesbocchio_jasypt-spring-...,passed,andvari-v2,v2
2,codex,0,210,0,NaN,ulisesbocchio_jasypt-spring-boot,20260505T122153Z__ulisesbocchio_jasypt-spring-...,passed,andvari-v3,v3
3,codex,0,156,0,NaN,mbechler_marshalsec,20260505T150455Z__mbechler_marshalsec__243c5aa8,passed,andvari,generated
4,codex,0,156,0,NaN,mbechler_marshalsec,20260505T150455Z__mbechler_marshalsec__243c5aa8,passed,andvari-v2,v2


## Derive enforcement fields

An invocation is counted as denied or blocked when either:

- `proxy_denied_count > 0`, or
- `blocked_egress_line_count > 0`.


In [5]:
summary = df[required].copy()

for col in ['proxy_access_line_count', 'proxy_denied_count', 'blocked_egress_line_count']:
    summary[col] = pd.to_numeric(summary[col], errors='coerce').fillna(0).astype(int)

summary['denied_or_blocked'] = (
    (summary['proxy_denied_count'] > 0) |
    (summary['blocked_egress_line_count'] > 0)
)

def classify_event_type(row):
    proxy = row['proxy_denied_count'] > 0
    firewall = row['blocked_egress_line_count'] > 0
    if proxy and firewall:
        return 'proxy_denial_and_firewall_block'
    if proxy:
        return 'proxy_denial'
    if firewall:
        return 'firewall_block'
    return 'none'

summary['main_event_type'] = summary.apply(classify_event_type, axis=1)

variant_labels = {
    'generated': 'Best-effort',
    'v2': 'Slightly degraded',
    'v3': 'Moderately degraded',
}
summary['variant_label'] = summary['variant'].map(variant_labels).fillna(summary['variant'])

summary = summary[
    [
        'run_id',
        'repo_slug',
        'agent',
        'variant',
        'variant_label',
        'step',
        'status',
        'reason',
        'proxy_access_line_count',
        'proxy_denied_count',
        'blocked_egress_line_count',
        'denied_or_blocked',
        'main_event_type',
    ]
]

summary.head()


,run_id,repo_slug,agent,variant,variant_label,step,status,reason,proxy_access_line_count,proxy_denied_count,blocked_egress_line_count,denied_or_blocked,main_event_type
0,20260505T122153Z__ulisesbocchio_jasypt-spring-...,ulisesbocchio_jasypt-spring-boot,codex,generated,Best-effort,andvari,passed,NaN,216,0,0,False,none
1,20260505T122153Z__ulisesbocchio_jasypt-spring-...,ulisesbocchio_jasypt-spring-boot,codex,v2,Slightly degraded,andvari-v2,passed,NaN,222,0,0,False,none
2,20260505T122153Z__ulisesbocchio_jasypt-spring-...,ulisesbocchio_jasypt-spring-boot,codex,v3,Moderately degraded,andvari-v3,passed,NaN,210,0,0,False,none
3,20260505T150455Z__mbechler_marshalsec__243c5aa8,mbechler_marshalsec,codex,generated,Best-effort,andvari,passed,NaN,156,0,0,False,none
4,20260505T150455Z__mbechler_marshalsec__243c5aa8,mbechler_marshalsec,codex,v2,Slightly degraded,andvari-v2,passed,NaN,156,0,0,False,none


## Export derived CSV

The derived artifact keeps one row per Andvari invocation, but removes unrelated operational fields.

In [6]:
summary.to_csv(OUTPUT_CSV, index=False)
OUTPUT_CSV


PosixPath('andvari_network_enforcement_summary.csv')

## Thesis summary table: agent by variant

This table corresponds to the compact Option B presentation.

In [7]:
table = (
    summary
    .groupby(['agent', 'variant_label'], dropna=False)
    .agg(
        invocations=('run_id', 'count'),
        denied_blocked_invocations=('denied_or_blocked', 'sum'),
    )
    .reset_index()
)

table['share'] = table['denied_blocked_invocations'] / table['invocations']
table['share_percent'] = (table['share'] * 100).round(2)

agent_order = {'claude': 0, 'Claude': 0, 'codex': 1, 'Codex': 1}
variant_order = {
    'Best-effort': 0,
    'Slightly degraded': 1,
    'Moderately degraded': 2,
}

table['_agent_order'] = table['agent'].map(agent_order).fillna(99)
table['_variant_order'] = table['variant_label'].map(variant_order).fillna(99)
table = table.sort_values(['_agent_order', '_variant_order']).drop(columns=['_agent_order', '_variant_order'])

table


,agent,variant_label,invocations,denied_blocked_invocations,share,share_percent
0,claude,Best-effort,40,5,0.125,12.5
2,claude,Slightly degraded,40,2,0.050,5.0
1,claude,Moderately degraded,40,3,0.075,7.5
3,codex,Best-effort,40,0,0.000,0.0
5,codex,Slightly degraded,40,2,0.050,5.0
4,codex,Moderately degraded,40,1,0.025,2.5


## Totals and validation checks

These checks make the expected denominator and event count explicit.

In [8]:
total_invocations = len(summary)
total_denied_blocked = int(summary['denied_or_blocked'].sum())
proxy_denial_invocations = int((summary['proxy_denied_count'] > 0).sum())
firewall_block_invocations = int((summary['blocked_egress_line_count'] > 0).sum())

print(f'Total invocations: {total_invocations}')
print(f'Denied/blocked invocations: {total_denied_blocked}')
print(f'Invocations with proxy denial: {proxy_denial_invocations}')
print(f'Invocations with firewall block: {firewall_block_invocations}')

# Expected values for the current thesis dataset.
assert total_invocations == 240
assert total_denied_blocked == 13


Total invocations: 240
Denied/blocked invocations: 13
Invocations with proxy denial: 11
Invocations with firewall block: 2


## Generate LaTeX table

This cell prints a LaTeX table that can be pasted into the thesis.

In [9]:
def latex_escape(value):
    return str(value).replace('_', r'\_')

lines = []
lines.append(r'\begin{table}[htbp]')
lines.append(r'\centering')
lines.append(r'\caption{Andvari invocations with denied proxy requests or firewall-blocked egress events by agent and reconstruction variant.}')
lines.append(r'\label{tab:proxy-firewall-events}')
lines.append(r'\begin{tabular}{llrrr}')
lines.append(r'\toprule')
lines.append(r'Agent & Variant & Invocations & Denied/blocked invocations & Share \\')
lines.append(r'\midrule')

previous_agent = None
for _, row in table.iterrows():
    agent = latex_escape(row['agent']).capitalize()
    variant = latex_escape(row['variant_label'])
    invocations = int(row['invocations'])
    denied = int(row['denied_blocked_invocations'])
    share = f"{row['share_percent']:.2f}\\%"

    if previous_agent is not None and agent != previous_agent:
        lines.append(r'\midrule')

    lines.append(f'{agent} & {variant} & {invocations} & {denied} & {share} \\\\')
    previous_agent = agent

total_share = total_denied_blocked / total_invocations * 100
lines.append(r'\midrule')
lines.append(f'Total & -- & {total_invocations} & {total_denied_blocked} & {total_share:.2f}\\% \\\\')
lines.append(r'\bottomrule')
lines.append(r'\end{tabular}')
lines.append(r'\end{table}')

latex_table = '\n'.join(lines)
print(latex_table)


\begin{table}[htbp]
\centering
\caption{Andvari invocations with denied proxy requests or firewall-blocked egress events by agent and reconstruction variant.}
\label{tab:proxy-firewall-events}
\begin{tabular}{llrrr}
\toprule
Agent & Variant & Invocations & Denied/blocked invocations & Share \\
\midrule
Claude & Best-effort & 40 & 5 & 12.50\% \\
Claude & Slightly degraded & 40 & 2 & 5.00\% \\
Claude & Moderately degraded & 40 & 3 & 7.50\% \\
\midrule
Codex & Best-effort & 40 & 0 & 0.00\% \\
Codex & Slightly degraded & 40 & 2 & 5.00\% \\
Codex & Moderately degraded & 40 & 1 & 2.50\% \\
\midrule
Total & -- & 240 & 13 & 5.42\% \\
\bottomrule
\end{tabular}
\end{table}
